In [1]:
import sys
sys.path.append("..")

from pathlib import Path
import numpy as np
import pandas as pd

from src.credit.features import TARGET, build_features

DATA_DIR = Path("../data/raw")
df = pd.read_csv(DATA_DIR / "application_train.csv")

X = build_features(df)
y = df[TARGET]
X.shape, y.shape, y.mean()

((307511, 92), (307511,), np.float64(0.08072881945686496))

In [2]:
numeric_cols = X.select_dtypes("number").columns.tolist()
categorical_cols = X.select_dtypes("object").columns.tolist()
len(numeric_cols), len(categorical_cols)

C:\Users\ibrah\AppData\Local\Temp\ipykernel_6768\2564670927.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes("object").columns.tolist()


(77, 15)

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42,
)
y_train.mean(), y_test.mean()

(np.float64(0.08072908198107379), np.float64(0.08072776937710356))

In [4]:
from src.credit.model import build_pipeline

pipe = build_pipeline(numeric_cols, categorical_cols)
pipe.fit(X_train, y_train)

n_features = pipe.named_steps["prep"].transform(X_train.head(1)).shape[1]
n_features

254

In [5]:
import collections

clf = pipe.named_steps["clf"]
prep = pipe.named_steps["prep"]
print("iterasi terpakai :", clf.n_iter_, "dari max_iter", clf.max_iter)

names = prep.get_feature_names_out()
blocks = collections.Counter(n.split("__")[0] for n in names)
n_ind = sum("missingindicator" in n for n in names)

print(f"total fitur      : {len(names)}")
print(f"  numerik asli   : {blocks['num'] - n_ind}")
print(f"  indikator NaN  : {n_ind}")
print(f"  one-hot        : {blocks['cat']}")

iterasi terpakai : [75] dari max_iter 1000
total fitur      : 254
  numerik asli   : 77
  indikator NaN  : 34
  one-hot        : 143


## Baseline pipeline

Every fitted step lives inside the pipeline, so imputation medians, scaling statistics and the one-hot vocabulary are learned from training rows only. `build_features` runs before the split because it is stateless — it drops columns and changes units, computing no statistic from the data.

Three choices worth stating:

**`add_indicator=True` on numeric imputation.** `EXT_SOURCE_1` is missing for 56% of applicants and missingness itself separates risk (6.91% vs 8.57% default). Median imputation without the flag would discard that signal.

**Missing categoricals become their own `MISSING` level, not the mode.**
Filling with the most common category asserts a value the applicant never gave.

**No `class_weight="balanced"`, deliberately.** Re-weighting to a 50/50 population leaves rank-ordering untouched (AUC and Gini are rank-based) but destroys calibration: predicted probabilities would estimate default under a population that does not exist, and come out systematically too high. Expected loss is PD x LGD x EAD, so an inflated PD mis-prices everything downstream in B8 and M2. The natural 8.07% prior is kept and the approve/decline cut-off is moved instead — which is B8's job, not the model's.